# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_10 — Turnover, Capacity, and Alpha Decay

### Research question

How do turnover, transaction costs, market impact, capacity limits, and alpha decay determine whether a statistically attractive signal can actually be traded at meaningful scale?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
05_06_performance_haircut_and_deflated_sharpe
05_07_purged_kfold_and_embargo_cv
05_08_probability_of_backtest_overfitting
05_09_white_reality_check_bootstrap
```

The earlier Phase 5 notebooks tested whether research results survive validation and multiple-testing correction. This notebook asks a practical implementation question:

> Can the alpha survive turnover, trading costs, market impact, limited liquidity, and delay?

It covers:

1. turnover definitions;
2. one-way and two-way turnover;
3. holding-period diagnostics;
4. signal autocorrelation;
5. information coefficient decay;
6. alpha half-life estimation;
7. execution delay and alpha decay;
8. spread and slippage costs;
9. linear impact model;
10. square-root impact model;
11. participation-rate constraints;
12. capacity curves by AUM;
13. breakeven AUM;
14. asset-level capacity bottlenecks;
15. turnover decomposition by signal component;
16. cost-adjusted IC;
17. cost-adjusted portfolio performance;
18. rebalance-frequency trade-off;
19. slow-alpha versus fast-alpha comparison;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> A signal is not scalable because it predicts returns. It is scalable only if the expected alpha arrives slowly enough, trades cheaply enough, and survives at the desired capital.

## 1. Why turnover and capacity matter

A strategy can have high gross alpha but still be unattractive if:

$$
NetAlpha = GrossAlpha - TradingCosts - Impact - Financing
$$

If turnover is high, every rebalance pays spread, slippage, and impact.

If the strategy trades too much relative to market volume, its own orders move prices.

If alpha decays quickly, execution delay destroys the signal before it can be monetised.

The practical question is:

$$
Capacity = \max AUM \quad \text{such that} \quad NetPerformance(AUM) \text{ remains acceptable}
$$

## 2. Turnover definitions

For portfolio weights $w_{i,t}$, one-way turnover is:

$$
TO_t = \frac{1}{2}\sum_i |w_{i,t}-w_{i,t-1}|
$$

Two-way turnover is:

$$
2TO_t = \sum_i |w_{i,t}-w_{i,t-1}|
$$

Many portfolio backtests use the two-way definition when applying cost to traded notional.

This notebook reports both.

## 3. Alpha decay

If expected alpha decays exponentially:

$$
\alpha(\Delta t) = \alpha_0 e^{-k\Delta t}
$$

then half-life $h$ satisfies:

$$
\frac{1}{2} = e^{-kh}
$$

so:

$$
h = \frac{\ln 2}{k}
$$

Short half-life signals need fast execution and low latency.

Long half-life signals can tolerate slower, cheaper trading.

## 4. Capacity intuition

Suppose daily traded notional is:

$$
Q_t = AUM \cdot \sum_i |\Delta w_{i,t}|
$$

Participation for asset $i$:

$$
p_{i,t} = \frac{AUM \cdot |\Delta w_{i,t}|}{ADV_{i,t}}
$$

A common square-root impact model:

$$
Impact_{i,t}^{bps} = Y\sigma_{i,t}^{daily,bps} \sqrt{p_{i,t}}
$$

As AUM rises, impact grows nonlinearly and net alpha falls.

Capacity is not a fixed property of a strategy. It depends on liquidity, turnover, volatility, execution style, and acceptable performance degradation.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class TurnoverCapacityConfig:
    n_days: int = 2_000
    annualisation: int = 252
    seed: int = 42
    gross_target: float = 1.20
    max_abs_weight: float = 0.25
    base_aum: float = 25_000_000.0
    aum_grid: tuple = (1e6, 2.5e6, 5e6, 10e6, 25e6, 50e6, 100e6, 250e6, 500e6, 1e9)
    rebalance_grid: tuple = (1, 2, 5, 10, 21)
    spread_cost_fraction: float = 0.50
    fixed_slippage_bps: float = 1.0
    commission_bps: float = 0.35
    linear_impact_coefficient_bps: float = 15.0
    sqrt_impact_coefficient: float = 0.65
    max_participation_guideline: float = 0.10
    alpha_decay_lags: tuple = (1, 2, 3, 5, 10, 21, 42, 63)
    min_net_sharpe_capacity: float = 0.50
    max_cost_drag_capacity: float = 0.05
    cvar_alpha: float = 0.95

config = TurnoverCapacityConfig()
config

## 6. Simulate multi-asset market and liquidity data

We simulate:

- daily returns;
- price series;
- spreads;
- average daily dollar volume;
- volatility regimes;
- asset metadata.

The assets deliberately differ in liquidity so capacity bottlenecks are visible.

In [ ]:
def simulate_market_and_liquidity(config: TurnoverCapacityConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2016-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    adv_usd = pd.Series({
        "US_EQ": 9e9,
        "EU_EQ": 4e9,
        "EM_EQ": 1.5e9,
        "US_BOND": 12e9,
        "EU_BOND": 6e9,
        "GOLD": 3e9,
        "OIL": 2.5e9,
        "COPPER": 1.0e9,
        "FX_CARRY": 2.5e9,
        "TREND": 0.7e9,
        "REIT": 1.5e9,
        "BTC_PROXY": 0.8e9,
    })

    base_spread_bps = pd.Series({
        "US_EQ": 1.0,
        "EU_EQ": 1.5,
        "EM_EQ": 4.5,
        "US_BOND": 0.7,
        "EU_BOND": 1.0,
        "GOLD": 2.0,
        "OIL": 3.5,
        "COPPER": 3.8,
        "FX_CARRY": 2.8,
        "TREND": 5.5,
        "REIT": 3.2,
        "BTC_PROXY": 14.0,
    })

    ann_mu = pd.Series({
        "US_EQ": 0.070,
        "EU_EQ": 0.060,
        "EM_EQ": 0.080,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    ann_vol = pd.Series({
        "US_EQ": 0.16,
        "EU_EQ": 0.18,
        "EM_EQ": 0.25,
        "US_BOND": 0.06,
        "EU_BOND": 0.07,
        "GOLD": 0.18,
        "OIL": 0.32,
        "COPPER": 0.28,
        "FX_CARRY": 0.10,
        "TREND": 0.12,
        "REIT": 0.20,
        "BTC_PROXY": 0.70,
    })

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.006, 0.007, 0.035])
    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.15,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.15,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    returns = np.zeros((config.n_days, len(assets)))
    factor_returns = np.zeros((config.n_days, len(factor_names)))

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_multiplier**2)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.014:
            stress_type[t] = "liquidity_shock"
            f[0] -= rng.uniform(0.010, 0.040)
            f[2] += rng.uniform(0.010, 0.060)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=6, size=len(assets)) * np.sqrt((6 - 2) / 6)
        eps = eps * ann_vol.values / np.sqrt(config.annualisation) * 0.35 * vol_multiplier

        returns[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns = pd.DataFrame(returns, index=dates, columns=assets)
    prices = 100.0 * np.exp(np.log1p(returns).cumsum())

    regime_s = pd.Series(regime, index=dates)
    spread_noise = rng.lognormal(0.0, 0.25, size=returns.shape)
    spread_bps = pd.DataFrame(
        spread_noise * base_spread_bps.values,
        index=dates,
        columns=assets,
    ).multiply(1 + 0.8 * regime_s, axis=0)

    volume_noise = rng.lognormal(0.0, 0.50, size=returns.shape)
    daily_volume_usd = pd.DataFrame(
        volume_noise * adv_usd.values,
        index=dates,
        columns=assets,
    ).multiply(1 - 0.25 * regime_s, axis=0)

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "adv_usd": [adv_usd[a] for a in assets],
        "base_spread_bps": [base_spread_bps[a] for a in assets],
        "ann_mu_assumption": [ann_mu[a] for a in assets],
        "ann_vol_assumption": [ann_vol[a] for a in assets],
    })

    regime_info = pd.DataFrame({
        "regime": regime,
        "stress_type": stress_type,
    }, index=dates)

    factors = pd.DataFrame(factor_returns, index=dates, columns=factor_names)

    return returns, prices, spread_bps, daily_volume_usd, metadata, regime_info, factors

returns, prices, spread_bps, daily_volume_usd, metadata, regime_info, factors = simulate_market_and_liquidity(config)
assets = metadata["asset"].tolist()

returns.head(), metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices.index, prices[asset], label=asset, alpha=0.75)
plt.title("Synthetic Multi-Asset Prices")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Regime Indicator")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Build slow and fast alpha signals

We create two signal families:

### Slow alpha

- longer lookbacks;
- more persistent;
- lower turnover;
- slower decay.

### Fast alpha

- shorter lookbacks;
- faster reaction;
- higher turnover;
- faster decay.

The comparison is the heart of this notebook.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def build_signal_components(prices, returns):
    components = {}

    components["momentum_21"] = cross_sectional_zscore(prices.pct_change(21))
    components["momentum_63"] = cross_sectional_zscore(prices.pct_change(63))
    components["momentum_126"] = cross_sectional_zscore(prices.pct_change(126))

    components["reversal_3"] = cross_sectional_zscore(-prices.pct_change(3))
    components["reversal_5"] = cross_sectional_zscore(-prices.pct_change(5))
    components["reversal_10"] = cross_sectional_zscore(-prices.pct_change(10))

    vol_21 = returns.rolling(21).std() * np.sqrt(config.annualisation)
    vol_63 = returns.rolling(63).std() * np.sqrt(config.annualisation)

    components["low_vol_21"] = cross_sectional_zscore(-vol_21)
    components["low_vol_63"] = cross_sectional_zscore(-vol_63)

    return {k: v.clip(-3, 3).fillna(0.0) for k, v in components.items()}

components = build_signal_components(prices, returns)

slow_signal = (
    0.50 * components["momentum_126"]
    + 0.30 * components["momentum_63"]
    + 0.20 * components["low_vol_63"]
).clip(-3, 3).fillna(0.0)

fast_signal = (
    0.45 * components["momentum_21"]
    + 0.35 * components["reversal_3"]
    + 0.20 * components["low_vol_21"]
).clip(-3, 3).fillna(0.0)

signals = {
    "slow_alpha": slow_signal,
    "fast_alpha": fast_signal,
}

slow_signal.tail()

In [ ]:
plt.figure(figsize=(12, 5))
for asset in ["US_EQ", "US_BOND", "GOLD", "OIL", "BTC_PROXY"]:
    plt.plot(slow_signal.index, slow_signal[asset], label=asset, alpha=0.8)
plt.title("Slow Alpha Signal")
plt.xlabel("Date")
plt.ylabel("Signal")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
for asset in ["US_EQ", "US_BOND", "GOLD", "OIL", "BTC_PROXY"]:
    plt.plot(fast_signal.index, fast_signal[asset], label=asset, alpha=0.8)
plt.title("Fast Alpha Signal")
plt.xlabel("Date")
plt.ylabel("Signal")
plt.legend()
plt.show()

## 8. Signal-to-weight engine

Signals become weights using:

1. cross-sectional demeaning;
2. inverse-vol scaling;
3. gross exposure normalisation;
4. max single-name weight cap;
5. rebalance schedule.

This isolates turnover impact by changing rebalance frequency.

In [ ]:
def signal_to_weights(signal, returns, gross_target, max_abs_weight, vol_lookback=63):
    vol = returns.rolling(vol_lookback).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(gross_target).fillna(0.0)

    weights = weights.clip(-max_abs_weight, max_abs_weight)
    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(gross_target).fillna(0.0)
    return weights

def apply_rebalance(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

weights_by_signal_rebalance = {}

for signal_name, signal in signals.items():
    for step in config.rebalance_grid:
        raw_w = signal_to_weights(signal, returns, config.gross_target, config.max_abs_weight)
        scheduled_w = apply_rebalance(raw_w, step)
        weights_by_signal_rebalance[(signal_name, step)] = scheduled_w

weights_by_signal_rebalance[("slow_alpha", 5)].tail()

## 9. Turnover diagnostics

We compute:

- one-way turnover;
- two-way turnover;
- annualised turnover;
- average holding-period proxy;
- gross and net exposure.

Holding-period proxy:

$$
HoldingPeriod \approx \frac{GrossExposure}{TwoWayTurnover}
$$

This is rough but useful.

In [ ]:
def turnover_diagnostics(weights):
    held = weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)

    two_way_turnover = delta.abs().sum(axis=1)
    one_way_turnover = 0.5 * two_way_turnover

    gross_exposure = held.abs().sum(axis=1)
    net_exposure = held.sum(axis=1)

    avg_two_way = two_way_turnover.mean()
    avg_one_way = one_way_turnover.mean()
    ann_two_way = avg_two_way * config.annualisation
    ann_one_way = avg_one_way * config.annualisation

    holding_period_days = gross_exposure.mean() / avg_two_way if avg_two_way > 0 else np.nan

    return {
        "avg_daily_one_way_turnover": avg_one_way,
        "avg_daily_two_way_turnover": avg_two_way,
        "annualised_one_way_turnover": ann_one_way,
        "annualised_two_way_turnover": ann_two_way,
        "holding_period_proxy_days": holding_period_days,
        "mean_gross_exposure": gross_exposure.mean(),
        "mean_net_exposure": net_exposure.mean(),
        "p95_two_way_turnover": two_way_turnover.quantile(0.95),
        "max_two_way_turnover": two_way_turnover.max(),
    }

turnover_rows = []

for (signal_name, step), weights in weights_by_signal_rebalance.items():
    row = {
        "signal_name": signal_name,
        "rebalance_step": step,
        **turnover_diagnostics(weights),
    }
    turnover_rows.append(row)

turnover_report = pd.DataFrame(turnover_rows).sort_values(["signal_name", "rebalance_step"])

turnover_report

In [ ]:
plt.figure(figsize=(10, 6))
for signal_name, sub in turnover_report.groupby("signal_name"):
    plt.plot(sub["rebalance_step"], sub["annualised_two_way_turnover"], marker="o", label=signal_name)
plt.title("Annualised Turnover vs Rebalance Frequency")
plt.xlabel("Rebalance step days")
plt.ylabel("Annualised two-way turnover")
plt.legend()
plt.show()

## 10. Signal autocorrelation

Slow signals should be more persistent.

Signal autocorrelation is a simple proxy for turnover pressure and alpha decay.

In [ ]:
def signal_autocorrelation(signal, lags):
    rows = []
    for lag in lags:
        corrs = []
        for asset in signal.columns:
            x = signal[asset]
            corr = x.corr(x.shift(lag))
            if np.isfinite(corr):
                corrs.append(corr)
        rows.append({
            "lag": lag,
            "mean_autocorr": np.mean(corrs),
            "median_autocorr": np.median(corrs),
        })
    return pd.DataFrame(rows)

signal_autocorr_frames = []

for name, signal in signals.items():
    ac = signal_autocorrelation(signal, list(config.alpha_decay_lags))
    ac["signal_name"] = name
    signal_autocorr_frames.append(ac)

signal_autocorr = pd.concat(signal_autocorr_frames, ignore_index=True)

signal_autocorr

In [ ]:
plt.figure(figsize=(10, 6))
for name, sub in signal_autocorr.groupby("signal_name"):
    plt.plot(sub["lag"], sub["mean_autocorr"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Signal Autocorrelation Decay")
plt.xlabel("Lag days")
plt.ylabel("Mean autocorrelation")
plt.legend()
plt.show()

## 11. Information coefficient decay

For each signal, compute cross-sectional IC at future horizons:

$$
IC_h = corr(signal_t, r_{t+h})
$$

The decay of IC over horizon gives a practical alpha half-life estimate.

In [ ]:
def mean_cross_sectional_ic(signal, returns, horizon):
    future_returns = returns.shift(-horizon)
    ics = []
    for date in signal.index:
        s = signal.loc[date]
        r = future_returns.loc[date]
        valid = s.notna() & r.notna()
        if valid.sum() >= 3:
            corr = s[valid].corr(r[valid])
            if np.isfinite(corr):
                ics.append(corr)
    return float(np.mean(ics)) if len(ics) else np.nan

ic_decay_rows = []

for signal_name, signal in signals.items():
    for h in config.alpha_decay_lags:
        ic_decay_rows.append({
            "signal_name": signal_name,
            "horizon_days": h,
            "mean_ic": mean_cross_sectional_ic(signal, returns, h),
        })

ic_decay = pd.DataFrame(ic_decay_rows)

ic_decay

In [ ]:
plt.figure(figsize=(10, 6))
for name, sub in ic_decay.groupby("signal_name"):
    plt.plot(sub["horizon_days"], sub["mean_ic"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Information Coefficient Decay")
plt.xlabel("Forward horizon days")
plt.ylabel("Mean cross-sectional IC")
plt.legend()
plt.show()

## 12. Estimate alpha half-life

We fit a rough exponential decay:

$$
IC_h \approx IC_0 e^{-kh}
$$

Then:

$$
half\_life = \frac{\ln 2}{k}
$$

This is approximate. It is a research diagnostic, not a law of nature.

In [ ]:
def estimate_half_life(ic_table):
    rows = []

    for signal_name, sub in ic_table.groupby("signal_name"):
        sub = sub.dropna().copy()
        sub = sub[sub["mean_ic"] > 0]

        if len(sub) < 3:
            rows.append({
                "signal_name": signal_name,
                "decay_rate_k": np.nan,
                "half_life_days": np.nan,
                "fit_quality_note": "insufficient_positive_ic",
            })
            continue

        x = sub["horizon_days"].to_numpy(dtype=float)
        y = np.log(sub["mean_ic"].to_numpy(dtype=float))

        X = np.column_stack([np.ones(len(x)), -x])
        beta = np.linalg.pinv(X.T @ X) @ X.T @ y
        intercept, k = beta

        fitted = X @ beta
        ss_res = ((y - fitted) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        half_life = np.log(2) / k if k > 0 else np.nan

        rows.append({
            "signal_name": signal_name,
            "decay_rate_k": k,
            "half_life_days": half_life,
            "log_ic_fit_r2": r2,
            "initial_ic_estimate": np.exp(intercept),
            "fit_quality_note": "ok",
        })

    return pd.DataFrame(rows)

alpha_half_life = estimate_half_life(ic_decay)

alpha_half_life

## 13. Cost model

Cost components per traded notional:

$$
\begin{aligned}
Cost^{bps}_{i,t} &= \frac{spread_{i,t}}{2} \\
&\quad + commission \\
&\quad + slippage \\
&\quad + impact
\end{aligned}
$$

Impact models:

### Linear

$$
impact = \eta p
$$

### Square-root

$$
impact = Y\sigma\sqrt{p}
$$

where:

$$
p = \frac{AUM|\Delta w|}{ADV}
$$

In [ ]:
def realised_vol_ann(returns, window=21):
    vol = returns.rolling(window).std().shift(1) * np.sqrt(config.annualisation)
    vol = vol.fillna(returns.expanding().std().shift(1) * np.sqrt(config.annualisation))
    vol = vol.fillna(returns.std() * np.sqrt(config.annualisation))
    return vol

vol_ann = realised_vol_ann(returns, 21)

def participation_matrix(delta_weights, aum, daily_volume_usd):
    traded_notional = delta_weights.abs() * aum
    return traded_notional / daily_volume_usd.replace(0.0, np.nan)

def cost_bps_matrix(delta_weights, aum, model="sqrt"):
    part = participation_matrix(delta_weights, aum, daily_volume_usd).fillna(0.0)
    daily_vol_bps = vol_ann / np.sqrt(config.annualisation) * 10000.0

    half_spread = spread_bps / 2.0
    explicit = config.commission_bps + config.fixed_slippage_bps

    if model == "linear":
        impact = config.linear_impact_coefficient_bps * part
    elif model == "sqrt":
        impact = config.sqrt_impact_coefficient * daily_vol_bps * np.sqrt(part.clip(lower=0.0))
    else:
        raise ValueError("model must be linear or sqrt")

    return half_spread + explicit + impact

def backtest_with_costs(weights, returns, aum, model="sqrt"):
    held = weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)

    gross_return = (held * returns).sum(axis=1)

    c_bps = cost_bps_matrix(delta, aum, model=model)
    trading_cost = (delta.abs() * c_bps / 10000.0).sum(axis=1)

    net_return = gross_return - trading_cost
    nav = (1 + net_return).cumprod()

    return pd.DataFrame({
        "gross_return": gross_return,
        "trading_cost": trading_cost,
        "net_return": net_return,
        "nav": nav,
        "two_way_turnover": delta.abs().sum(axis=1),
        "one_way_turnover": 0.5 * delta.abs().sum(axis=1),
        "gross_exposure": held.abs().sum(axis=1),
        "net_exposure": held.sum(axis=1),
    }, index=returns.index), c_bps, participation_matrix(delta, aum, daily_volume_usd)

# Example.
example_bt, example_cost_bps, example_participation = backtest_with_costs(
    weights_by_signal_rebalance[("fast_alpha", 5)],
    returns,
    config.base_aum,
    model="sqrt",
)

example_bt.head()

## 14. Performance metrics

We evaluate gross and net performance:

- annualised return;
- annualised volatility;
- Sharpe;
- max drawdown;
- CVaR;
- annualised cost drag;
- turnover.

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def performance_summary(bt, name, signal_name, rebalance_step, aum, impact_model):
    r = bt["net_return"].dropna()
    gross_r = bt["gross_return"].dropna()
    nav = (1 + r).cumprod()
    _, mdd = max_drawdown(nav)

    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    gross_ann_return = (1 + gross_r).prod() ** (config.annualisation / len(gross_r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    return {
        "strategy_name": name,
        "signal_name": signal_name,
        "rebalance_step": rebalance_step,
        "aum": aum,
        "impact_model": impact_model,
        "gross_annualised_return": gross_ann_return,
        "net_annualised_return": ann_return,
        "annualised_vol": ann_vol,
        "net_sharpe": sharpe,
        "max_drawdown": mdd,
        "VaR": var,
        "CVaR": cvar,
        "annualised_cost_drag": bt["trading_cost"].mean() * config.annualisation,
        "avg_daily_two_way_turnover": bt["two_way_turnover"].mean(),
        "annualised_two_way_turnover": bt["two_way_turnover"].mean() * config.annualisation,
        "mean_gross_exposure": bt["gross_exposure"].mean(),
        "total_return": nav.iloc[-1] - 1,
    }

performance_rows = []

for (signal_name, step), weights in weights_by_signal_rebalance.items():
    for model in ["linear", "sqrt"]:
        bt, _, _ = backtest_with_costs(weights, returns, config.base_aum, model=model)
        performance_rows.append(performance_summary(bt, f"{signal_name}_reb{step}_{model}", signal_name, step, config.base_aum, model))

performance_report = pd.DataFrame(performance_rows).sort_values("net_sharpe", ascending=False)

performance_report

In [ ]:
plt.figure(figsize=(10, 6))
for signal_name, sub in performance_report[performance_report["impact_model"] == "sqrt"].groupby("signal_name"):
    sub = sub.sort_values("rebalance_step")
    plt.plot(sub["rebalance_step"], sub["net_sharpe"], marker="o", label=signal_name)
plt.axhline(0, linestyle="--")
plt.title("Net Sharpe vs Rebalance Frequency, Square-Root Impact")
plt.xlabel("Rebalance step days")
plt.ylabel("Net Sharpe")
plt.legend()
plt.show()

## 15. Cost attribution by asset

We identify which assets consume most of the trading budget.

Cost by asset:

$$
Cost_{i,t} = |\Delta w_{i,t}| \frac{Cost^{bps}_{i,t}}{10000}
$$

In [ ]:
def cost_attribution_by_asset(weights, aum, model="sqrt"):
    held = weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)
    c_bps = cost_bps_matrix(delta, aum, model=model)

    cost_by_asset = delta.abs() * c_bps / 10000.0
    participation = participation_matrix(delta, aum, daily_volume_usd).fillna(0.0)

    rows = []
    for asset in weights.columns:
        rows.append({
            "asset": asset,
            "annualised_cost_contribution": cost_by_asset[asset].mean() * config.annualisation,
            "total_cost_contribution": cost_by_asset[asset].sum(),
            "mean_participation": participation[asset].mean(),
            "p95_participation": participation[asset].quantile(0.95),
            "max_participation": participation[asset].max(),
            "mean_cost_bps": c_bps[asset].mean(),
            "p95_cost_bps": c_bps[asset].quantile(0.95),
        })

    out = pd.DataFrame(rows).merge(metadata[["asset", "asset_class", "adv_usd", "base_spread_bps"]], on="asset", how="left")
    out["cost_share"] = out["total_cost_contribution"] / out["total_cost_contribution"].sum()
    return out.sort_values("total_cost_contribution", ascending=False)

selected_weights = weights_by_signal_rebalance[("fast_alpha", 5)]
cost_attr_fast = cost_attribution_by_asset(selected_weights, config.base_aum, "sqrt")

cost_attr_fast

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = cost_attr_fast.sort_values("annualised_cost_contribution")
plt.barh(plot_df["asset"], plot_df["annualised_cost_contribution"])
plt.title("Annualised Cost Contribution by Asset, Fast Alpha")
plt.xlabel("Annualised cost contribution")
plt.ylabel("Asset")
plt.show()

## 16. Capacity curves

For each AUM level, recompute participation, impact, cost, and net performance.

This creates strategy capacity curves.

In [ ]:
capacity_rows = []
capacity_detail = {}

for (signal_name, step), weights in weights_by_signal_rebalance.items():
    for aum in config.aum_grid:
        for model in ["linear", "sqrt"]:
            bt, c_bps, part = backtest_with_costs(weights, returns, aum, model=model)
            row = performance_summary(bt, f"{signal_name}_reb{step}_{model}", signal_name, step, aum, model)
            row["max_participation"] = part.max().max()
            row["p95_participation"] = part.stack().quantile(0.95)
            row["mean_cost_bps"] = c_bps.stack().mean()
            row["p95_cost_bps"] = c_bps.stack().quantile(0.95)
            capacity_rows.append(row)

capacity_curve = pd.DataFrame(capacity_rows)

capacity_curve.head()

In [ ]:
plt.figure(figsize=(10, 6))
for signal_name in ["slow_alpha", "fast_alpha"]:
    sub = capacity_curve[
        (capacity_curve["signal_name"] == signal_name)
        & (capacity_curve["rebalance_step"] == 5)
        & (capacity_curve["impact_model"] == "sqrt")
    ].sort_values("aum")
    plt.plot(sub["aum"], sub["net_sharpe"], marker="o", label=signal_name)
plt.xscale("log")
plt.axhline(config.min_net_sharpe_capacity, linestyle="--", label="capacity Sharpe threshold")
plt.title("Capacity Curve: Net Sharpe vs AUM")
plt.xlabel("AUM")
plt.ylabel("Net Sharpe")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for signal_name in ["slow_alpha", "fast_alpha"]:
    sub = capacity_curve[
        (capacity_curve["signal_name"] == signal_name)
        & (capacity_curve["rebalance_step"] == 5)
        & (capacity_curve["impact_model"] == "sqrt")
    ].sort_values("aum")
    plt.plot(sub["aum"], sub["annualised_cost_drag"], marker="o", label=signal_name)
plt.xscale("log")
plt.axhline(config.max_cost_drag_capacity, linestyle="--", label="cost drag limit")
plt.title("Capacity Curve: Cost Drag vs AUM")
plt.xlabel("AUM")
plt.ylabel("Annualised cost drag")
plt.legend()
plt.show()

## 17. Estimate breakeven capacity

Capacity rules:

1. net Sharpe must stay above threshold;
2. annual cost drag must stay below threshold;
3. p95 participation must stay below guideline.

The capacity estimate is the largest AUM satisfying all three.

In [ ]:
def estimate_capacity(capacity_curve):
    rows = []

    for (signal_name, step, model), sub in capacity_curve.groupby(["signal_name", "rebalance_step", "impact_model"]):
        sub = sub.sort_values("aum").copy()
        sub["passes_capacity_rules"] = (
            (sub["net_sharpe"] >= config.min_net_sharpe_capacity)
            & (sub["annualised_cost_drag"] <= config.max_cost_drag_capacity)
            & (sub["p95_participation"] <= config.max_participation_guideline)
        )

        passing = sub[sub["passes_capacity_rules"]]
        capacity = passing["aum"].max() if len(passing) else 0.0

        rows.append({
            "signal_name": signal_name,
            "rebalance_step": step,
            "impact_model": model,
            "estimated_capacity_aum": capacity,
            "n_passing_aum_points": len(passing),
            "max_tested_aum": sub["aum"].max(),
            "capacity_rule": "net_sharpe>=threshold and cost_drag<=threshold and p95_participation<=guideline",
        })

    return pd.DataFrame(rows).sort_values("estimated_capacity_aum", ascending=False)

capacity_estimates = estimate_capacity(capacity_curve)

capacity_estimates.head(20)

## 18. Asset-level capacity bottlenecks

At target AUM, the bottleneck is often one or two less-liquid assets.

We identify assets with high participation and high cost.

In [ ]:
def capacity_bottleneck_report(weights, aum, model="sqrt"):
    attr = cost_attribution_by_asset(weights, aum, model=model)
    attr["flag_p95_participation_above_guideline"] = attr["p95_participation"] > config.max_participation_guideline
    attr["flag_cost_share_above_25pct"] = attr["cost_share"] > 0.25
    attr["bottleneck_score"] = (
        attr["p95_participation"] / config.max_participation_guideline
        + 3.0 * attr["cost_share"]
    )
    return attr.sort_values("bottleneck_score", ascending=False)

bottleneck_fast_100m = capacity_bottleneck_report(
    weights_by_signal_rebalance[("fast_alpha", 5)],
    100_000_000,
    model="sqrt"
)

bottleneck_fast_100m

## 19. Alpha decay under execution delay

Expected alpha after delay:

$$
\alpha_{\Delta t} = \alpha_0 e^{-\ln 2 \Delta t/h}
$$

We estimate delay-adjusted return by reducing gross alpha while leaving costs unchanged.

This is approximate but useful for comparing fast and slow signals.

In [ ]:
def delay_adjusted_performance(bt, half_life_days, delay_days):
    decay = np.exp(-np.log(2) * delay_days / half_life_days) if np.isfinite(half_life_days) and half_life_days > 0 else 0.0

    # Approximation: scale gross return by decay, costs unchanged.
    adjusted_net = bt["gross_return"] * decay - bt["trading_cost"]
    adjusted_nav = (1 + adjusted_net).cumprod()

    out = bt.copy()
    out["delay_days"] = delay_days
    out["alpha_decay_multiplier"] = decay
    out["delay_adjusted_net_return"] = adjusted_net
    out["delay_adjusted_nav"] = adjusted_nav
    return out

delay_rows = []

for signal_name in ["slow_alpha", "fast_alpha"]:
    hl = alpha_half_life[alpha_half_life["signal_name"] == signal_name]["half_life_days"].iloc[0]
    weights = weights_by_signal_rebalance[(signal_name, 5)]
    bt, _, _ = backtest_with_costs(weights, returns, config.base_aum, model="sqrt")

    for delay in [0, 1, 2, 3, 5, 10, 21]:
        adj = delay_adjusted_performance(bt, hl, delay)
        r = adj["delay_adjusted_net_return"]
        ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
        ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
        sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

        delay_rows.append({
            "signal_name": signal_name,
            "half_life_days": hl,
            "delay_days": delay,
            "alpha_decay_multiplier": adj["alpha_decay_multiplier"].iloc[0],
            "delay_adjusted_ann_return": ann_return,
            "delay_adjusted_vol": ann_vol,
            "delay_adjusted_sharpe": sharpe,
        })

delay_impact_report = pd.DataFrame(delay_rows)

delay_impact_report

In [ ]:
plt.figure(figsize=(10, 6))
for signal_name, sub in delay_impact_report.groupby("signal_name"):
    plt.plot(sub["delay_days"], sub["delay_adjusted_sharpe"], marker="o", label=signal_name)
plt.axhline(0, linestyle="--")
plt.title("Delay-Adjusted Sharpe")
plt.xlabel("Execution delay days")
plt.ylabel("Delay-adjusted Sharpe")
plt.legend()
plt.show()

## 20. Turnover decomposition by signal component

We approximate how each signal component contributes to turnover.

For each component, we build weights using only that component and compare its turnover to the combined strategy.

In [ ]:
def component_turnover_decomposition(components, returns, signal_name, component_names, rebalance_step):
    rows = []

    # Combined signal used as reference.
    if signal_name == "slow_alpha":
        combined = slow_signal
    elif signal_name == "fast_alpha":
        combined = fast_signal
    else:
        raise ValueError("unknown signal")

    combined_w = apply_rebalance(signal_to_weights(combined, returns, config.gross_target, config.max_abs_weight), rebalance_step)
    combined_turnover = turnover_diagnostics(combined_w)["annualised_two_way_turnover"]

    for comp_name in component_names:
        comp_signal = components[comp_name]
        comp_w = apply_rebalance(signal_to_weights(comp_signal, returns, config.gross_target, config.max_abs_weight), rebalance_step)
        comp_turnover = turnover_diagnostics(comp_w)["annualised_two_way_turnover"]

        rows.append({
            "signal_name": signal_name,
            "component": comp_name,
            "rebalance_step": rebalance_step,
            "component_annualised_turnover": comp_turnover,
            "combined_annualised_turnover": combined_turnover,
            "turnover_ratio_vs_combined": comp_turnover / combined_turnover if combined_turnover > 0 else np.nan,
        })

    return pd.DataFrame(rows)

slow_component_turnover = component_turnover_decomposition(
    components,
    returns,
    "slow_alpha",
    ["momentum_126", "momentum_63", "low_vol_63"],
    5,
)

fast_component_turnover = component_turnover_decomposition(
    components,
    returns,
    "fast_alpha",
    ["momentum_21", "reversal_3", "low_vol_21"],
    5,
)

component_turnover = pd.concat([slow_component_turnover, fast_component_turnover], ignore_index=True)

component_turnover

## 21. Cost-adjusted IC

A signal may have positive IC but not enough to cover expected trading cost.

A rough cost-adjusted IC proxy:

$$
IC^{net}_h = IC_h - \lambda \cdot CostIntensity
$$

where cost intensity is annualised cost drag or turnover.

This is not a statistical identity. It is a useful research screening device.

In [ ]:
cost_adjusted_ic_rows = []

for signal_name in ["slow_alpha", "fast_alpha"]:
    ic1 = ic_decay[(ic_decay["signal_name"] == signal_name) & (ic_decay["horizon_days"] == 1)]["mean_ic"].iloc[0]
    weights = weights_by_signal_rebalance[(signal_name, 5)]
    bt, _, _ = backtest_with_costs(weights, returns, config.base_aum, model="sqrt")
    ann_cost = bt["trading_cost"].mean() * config.annualisation
    ann_turnover = bt["two_way_turnover"].mean() * config.annualisation

    cost_adjusted_ic_rows.append({
        "signal_name": signal_name,
        "one_day_ic": ic1,
        "annualised_cost_drag": ann_cost,
        "annualised_turnover": ann_turnover,
        "cost_adjusted_ic_proxy": ic1 - 0.10 * ann_cost - 0.005 * ann_turnover,
    })

cost_adjusted_ic = pd.DataFrame(cost_adjusted_ic_rows)

cost_adjusted_ic

## 22. Rebalance-frequency trade-off

Rebalancing faster can capture fresh alpha but increases turnover.

Rebalancing slower reduces costs but may miss alpha.

We compare net Sharpe across rebalance frequencies.

In [ ]:
rebalance_tradeoff = performance_report[
    (performance_report["impact_model"] == "sqrt")
].copy()

rebalance_tradeoff = rebalance_tradeoff[[
    "signal_name",
    "rebalance_step",
    "gross_annualised_return",
    "net_annualised_return",
    "net_sharpe",
    "annualised_cost_drag",
    "annualised_two_way_turnover",
    "max_drawdown",
]]

rebalance_tradeoff.sort_values(["signal_name", "rebalance_step"])

In [ ]:
plt.figure(figsize=(10, 6))
for signal_name, sub in rebalance_tradeoff.groupby("signal_name"):
    sub = sub.sort_values("rebalance_step")
    plt.plot(sub["rebalance_step"], sub["net_annualised_return"], marker="o", label=signal_name)
plt.axhline(0, linestyle="--")
plt.title("Net Annualised Return vs Rebalance Frequency")
plt.xlabel("Rebalance step days")
plt.ylabel("Net annualised return")
plt.legend()
plt.show()

## 23. Capacity stress during liquidity shocks

Capacity should be checked specifically during liquidity stress.

We compare participation and costs on normal versus liquidity-shock days.

In [ ]:
def stress_capacity_report(weights, aum, model="sqrt"):
    held = weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)

    part = participation_matrix(delta, aum, daily_volume_usd).fillna(0.0)
    c_bps = cost_bps_matrix(delta, aum, model=model)
    cost = delta.abs() * c_bps / 10000.0

    rows = []
    for stress_type, idx in regime_info.groupby("stress_type").groups.items():
        idx = list(idx)
        if len(idx) < 5:
            continue
        rows.append({
            "stress_type": stress_type,
            "n_days": len(idx),
            "mean_participation": part.loc[idx].stack().mean(),
            "p95_participation": part.loc[idx].stack().quantile(0.95),
            "max_participation": part.loc[idx].max().max(),
            "mean_cost_bps": c_bps.loc[idx].stack().mean(),
            "p95_cost_bps": c_bps.loc[idx].stack().quantile(0.95),
            "annualised_cost_drag_conditional": cost.loc[idx].sum(axis=1).mean() * config.annualisation,
        })

    return pd.DataFrame(rows).sort_values("p95_participation", ascending=False)

stress_capacity_fast = stress_capacity_report(
    weights_by_signal_rebalance[("fast_alpha", 5)],
    100_000_000,
    "sqrt"
)

stress_capacity_fast

## 24. Capacity dashboard

Combine all diagnostics into a dashboard:

- base AUM net performance;
- estimated capacity;
- half-life;
- turnover;
- p95 participation;
- bottleneck asset;
- delay sensitivity.

In [ ]:
dashboard_rows = []

for signal_name in ["slow_alpha", "fast_alpha"]:
    for step in [5, 10, 21]:
        cap_row = capacity_estimates[
            (capacity_estimates["signal_name"] == signal_name)
            & (capacity_estimates["rebalance_step"] == step)
            & (capacity_estimates["impact_model"] == "sqrt")
        ].iloc[0]

        perf_row = performance_report[
            (performance_report["signal_name"] == signal_name)
            & (performance_report["rebalance_step"] == step)
            & (performance_report["impact_model"] == "sqrt")
        ].iloc[0]

        half_life = alpha_half_life[alpha_half_life["signal_name"] == signal_name]["half_life_days"].iloc[0]

        weights = weights_by_signal_rebalance[(signal_name, step)]
        bottleneck = capacity_bottleneck_report(weights, 100_000_000, "sqrt").iloc[0]

        delay_5 = delay_impact_report[
            (delay_impact_report["signal_name"] == signal_name)
            & (delay_impact_report["delay_days"] == 5)
        ]["delay_adjusted_sharpe"].iloc[0]

        dashboard_rows.append({
            "signal_name": signal_name,
            "rebalance_step": step,
            "base_aum": config.base_aum,
            "base_net_sharpe": perf_row["net_sharpe"],
            "base_net_ann_return": perf_row["net_annualised_return"],
            "base_cost_drag": perf_row["annualised_cost_drag"],
            "estimated_capacity_aum": cap_row["estimated_capacity_aum"],
            "half_life_days": half_life,
            "annualised_turnover": perf_row["annualised_two_way_turnover"],
            "delay_5d_adjusted_sharpe": delay_5,
            "bottleneck_asset_at_100m": bottleneck["asset"],
            "bottleneck_asset_p95_participation": bottleneck["p95_participation"],
            "bottleneck_asset_cost_share": bottleneck["cost_share"],
        })

capacity_dashboard = pd.DataFrame(dashboard_rows).sort_values("estimated_capacity_aum", ascending=False)

capacity_dashboard

## 25. Governance flags

A strategy should be reviewed if:

1. annualised turnover is excessive;
2. holding period is too short for the estimated alpha half-life;
3. p95 participation exceeds guideline;
4. cost drag is too high;
5. capacity is below desired AUM;
6. delay-adjusted Sharpe collapses;
7. one asset dominates cost;
8. liquidity-shock cost is too high.

In [ ]:
desired_aum = 100_000_000

selected_dashboard = capacity_dashboard[
    (capacity_dashboard["signal_name"] == "fast_alpha")
    & (capacity_dashboard["rebalance_step"] == 5)
].iloc[0]

selected_capacity_row = capacity_curve[
    (capacity_curve["signal_name"] == "fast_alpha")
    & (capacity_curve["rebalance_step"] == 5)
    & (capacity_curve["impact_model"] == "sqrt")
    & (capacity_curve["aum"] == desired_aum)
]

if len(selected_capacity_row):
    selected_capacity_row = selected_capacity_row.iloc[0]
else:
    selected_capacity_row = capacity_curve[
        (capacity_curve["signal_name"] == "fast_alpha")
        & (capacity_curve["rebalance_step"] == 5)
        & (capacity_curve["impact_model"] == "sqrt")
    ].sort_values("aum").iloc[-1]

liquidity_shock_cost = stress_capacity_fast[
    stress_capacity_fast["stress_type"] == "liquidity_shock"
]["annualised_cost_drag_conditional"]

liquidity_shock_cost_value = float(liquidity_shock_cost.iloc[0]) if len(liquidity_shock_cost) else np.nan

governance_flags = pd.DataFrame([{
    "strategy": "fast_alpha_reb5_sqrt",
    "desired_aum": desired_aum,
    "estimated_capacity_aum": selected_dashboard["estimated_capacity_aum"],
    "base_net_sharpe": selected_dashboard["base_net_sharpe"],
    "base_cost_drag": selected_dashboard["base_cost_drag"],
    "annualised_turnover": selected_dashboard["annualised_turnover"],
    "half_life_days": selected_dashboard["half_life_days"],
    "delay_5d_adjusted_sharpe": selected_dashboard["delay_5d_adjusted_sharpe"],
    "p95_participation_at_desired_aum": selected_capacity_row["p95_participation"],
    "bottleneck_asset": selected_dashboard["bottleneck_asset_at_100m"],
    "bottleneck_asset_cost_share": selected_dashboard["bottleneck_asset_cost_share"],
    "liquidity_shock_annualised_cost_drag": liquidity_shock_cost_value,
    "flag_capacity_below_desired_aum": bool(selected_dashboard["estimated_capacity_aum"] < desired_aum),
    "flag_turnover_above_10x_ann": bool(selected_dashboard["annualised_turnover"] > 10.0),
    "flag_cost_drag_above_5pct": bool(selected_dashboard["base_cost_drag"] > 0.05),
    "flag_p95_participation_above_guideline": bool(selected_capacity_row["p95_participation"] > config.max_participation_guideline),
    "flag_delay_5d_sharpe_below_0": bool(selected_dashboard["delay_5d_adjusted_sharpe"] < 0),
    "flag_bottleneck_cost_share_above_35pct": bool(selected_dashboard["bottleneck_asset_cost_share"] > 0.35),
    "flag_liquidity_shock_cost_above_10pct": bool(liquidity_shock_cost_value > 0.10) if np.isfinite(liquidity_shock_cost_value) else False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_capacity_below_desired_aum",
        "flag_turnover_above_10x_ann",
        "flag_cost_drag_above_5pct",
        "flag_p95_participation_above_guideline",
        "flag_delay_5d_sharpe_below_0",
        "flag_bottleneck_cost_share_above_35pct",
        "flag_liquidity_shock_cost_above_10pct",
    ]
].any(axis=1)

governance_flags

## 26. Audit checks

Numerical and process checks:

1. weights sum to intended gross exposure when active;
2. turnover is non-negative;
3. participation is finite;
4. costs are non-negative;
5. capacity curve cost increases with AUM;
6. alpha decay multipliers decrease with delay;
7. estimated capacities are within tested range;
8. dashboard outputs are finite.

In [ ]:
audit_rows = []

sample_weights = weights_by_signal_rebalance[("fast_alpha", 5)]
gross = sample_weights.abs().sum(axis=1)
active_gross = gross[gross > 0]
gross_ok = bool((active_gross <= config.gross_target + 1e-8).all())
audit_rows.append({
    "check": "weights_gross_exposure_within_target",
    "value": float(active_gross.max()),
    "passed": gross_ok,
})

turnover_nonnegative = bool((performance_report["annualised_two_way_turnover"] >= -1e-12).all())
audit_rows.append({
    "check": "turnover_nonnegative",
    "value": turnover_nonnegative,
    "passed": turnover_nonnegative,
})

participation_finite = bool(np.isfinite(example_participation.fillna(0.0).to_numpy()).all())
audit_rows.append({
    "check": "participation_finite",
    "value": participation_finite,
    "passed": participation_finite,
})

costs_nonnegative = bool((example_cost_bps >= -1e-12).all().all())
audit_rows.append({
    "check": "cost_bps_nonnegative",
    "value": costs_nonnegative,
    "passed": costs_nonnegative,
})

capacity_monotonic_checks = []
for (signal_name, step, model), sub in capacity_curve.groupby(["signal_name", "rebalance_step", "impact_model"]):
    sub = sub.sort_values("aum")
    capacity_monotonic_checks.append(sub["annualised_cost_drag"].is_monotonic_increasing)
capacity_cost_monotone = bool(all(capacity_monotonic_checks))
audit_rows.append({
    "check": "capacity_cost_drag_monotone_in_aum",
    "value": capacity_cost_monotone,
    "passed": capacity_cost_monotone,
})

decay_monotone = True
for signal_name, sub in delay_impact_report.groupby("signal_name"):
    decay_monotone = decay_monotone and sub.sort_values("delay_days")["alpha_decay_multiplier"].is_monotonic_decreasing
audit_rows.append({
    "check": "alpha_decay_multiplier_decreases_with_delay",
    "value": bool(decay_monotone),
    "passed": bool(decay_monotone),
})

capacity_within_range = bool(
    (capacity_estimates["estimated_capacity_aum"] >= 0).all()
    and (capacity_estimates["estimated_capacity_aum"] <= max(config.aum_grid)).all()
)
audit_rows.append({
    "check": "estimated_capacity_within_tested_range",
    "value": capacity_within_range,
    "passed": capacity_within_range,
})

dashboard_finite = bool(np.isfinite(capacity_dashboard.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "capacity_dashboard_numeric_outputs_finite",
    "value": dashboard_finite,
    "passed": dashboard_finite,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 27. Practical checklist for turnover, capacity, and alpha decay

Before approving a strategy for scale:

1. **Measure turnover**  
   Report both one-way and two-way turnover.

2. **Estimate holding period**  
   Fast turnover must be justified by fast and strong alpha.

3. **Measure IC decay**  
   Know the signal half-life.

4. **Model execution delay**  
   Alpha that disappears after a small delay is fragile.

5. **Estimate participation**  
   Large AUM can make small weights impossible to trade.

6. **Use nonlinear impact**  
   Capacity rarely scales linearly.

7. **Identify bottleneck assets**  
   One illiquid asset can dominate costs.

8. **Stress liquidity regimes**  
   Capacity is lower when spreads widen and volumes fall.

9. **Check rebalance frequency**  
   Faster rebalancing is not automatically better.

10. **Define capacity rules in advance**  
   Sharpe, cost drag, participation and drawdown thresholds should be pre-agreed.

## 28. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/turnover_capacity_and_alpha_decay_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "returns": output_dir / "returns.csv",
    "prices": output_dir / "prices.csv",
    "spread_bps": output_dir / "spread_bps.csv",
    "daily_volume_usd": output_dir / "daily_volume_usd.csv",
    "metadata": output_dir / "metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "factors": output_dir / "factors.csv",
    "slow_signal": output_dir / "slow_signal.csv",
    "fast_signal": output_dir / "fast_signal.csv",
    "turnover_report": output_dir / "turnover_report.csv",
    "signal_autocorr": output_dir / "signal_autocorr.csv",
    "ic_decay": output_dir / "ic_decay.csv",
    "alpha_half_life": output_dir / "alpha_half_life.csv",
    "performance_report": output_dir / "performance_report.csv",
    "cost_attr_fast": output_dir / "cost_attribution_fast.csv",
    "capacity_curve": output_dir / "capacity_curve.csv",
    "capacity_estimates": output_dir / "capacity_estimates.csv",
    "bottleneck_fast_100m": output_dir / "bottleneck_fast_100m.csv",
    "delay_impact_report": output_dir / "delay_impact_report.csv",
    "component_turnover": output_dir / "component_turnover.csv",
    "cost_adjusted_ic": output_dir / "cost_adjusted_ic.csv",
    "rebalance_tradeoff": output_dir / "rebalance_tradeoff.csv",
    "stress_capacity_fast": output_dir / "stress_capacity_fast.csv",
    "capacity_dashboard": output_dir / "capacity_dashboard.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

returns.to_csv(paths["returns"])
prices.to_csv(paths["prices"])
spread_bps.to_csv(paths["spread_bps"])
daily_volume_usd.to_csv(paths["daily_volume_usd"])
metadata.to_csv(paths["metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
factors.to_csv(paths["factors"])
slow_signal.to_csv(paths["slow_signal"])
fast_signal.to_csv(paths["fast_signal"])
turnover_report.to_csv(paths["turnover_report"], index=False)
signal_autocorr.to_csv(paths["signal_autocorr"], index=False)
ic_decay.to_csv(paths["ic_decay"], index=False)
alpha_half_life.to_csv(paths["alpha_half_life"], index=False)
performance_report.to_csv(paths["performance_report"], index=False)
cost_attr_fast.to_csv(paths["cost_attr_fast"], index=False)
capacity_curve.to_csv(paths["capacity_curve"], index=False)
capacity_estimates.to_csv(paths["capacity_estimates"], index=False)
bottleneck_fast_100m.to_csv(paths["bottleneck_fast_100m"], index=False)
delay_impact_report.to_csv(paths["delay_impact_report"], index=False)
component_turnover.to_csv(paths["component_turnover"], index=False)
cost_adjusted_ic.to_csv(paths["cost_adjusted_ic"], index=False)
rebalance_tradeoff.to_csv(paths["rebalance_tradeoff"], index=False)
stress_capacity_fast.to_csv(paths["stress_capacity_fast"], index=False)
capacity_dashboard.to_csv(paths["capacity_dashboard"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

# Save representative weight matrices.
for (signal_name, step), weights in weights_by_signal_rebalance.items():
    if step in [1, 5, 21]:
        weights.to_csv(output_dir / f"weights_{signal_name}_rebalance_{step}.csv")

manifest = {
    "dataset_name": "turnover_capacity_and_alpha_decay_outputs",
    "schema_version": "turnover_capacity_and_alpha_decay_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "assets": assets,
    "capacity_dashboard": capacity_dashboard.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "estimated_half_life": alpha_half_life.to_dict(orient="records"),
    "notes": [
        "Slow and fast alpha signals are compared across rebalance frequencies.",
        "Turnover is reported as one-way and two-way turnover.",
        "IC decay is used to estimate approximate alpha half-life.",
        "Capacity curves recompute impact and participation at different AUM levels.",
        "Square-root and linear impact models are supported.",
        "Capacity is estimated using net Sharpe, cost drag and participation constraints.",
        "This notebook focuses on tradability and capacity, not alpha discovery."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["capacity_dashboard"], paths["capacity_curve"], paths["governance_flags"], paths["audit_report"], paths["manifest"]

## 29. Limitations

### Synthetic data

The liquidity and return data are synthetic. Real ADV, spread, volatility and market impact must come from historical and live execution data.

### Simple impact models

Linear and square-root impact are useful approximations but not full execution models.

### Approximate alpha decay

IC decay and exponential half-life are diagnostics, not guaranteed future behaviour.

### Weight-based accounting

The notebook uses weights, not contract/share-level execution.

### Capacity grid is discrete

True breakeven capacity lies between grid points and should be solved more precisely in production.

### No order scheduling

The notebook does not model TWAP, VWAP, POV or adaptive execution.

### No exchange constraints

Futures tick sizes, limits, margin, night sessions and exchange fees are not included here.

## 30. Research frontier and extensions

Important extensions include:

1. Almgren-Chriss optimal execution;
2. transient versus permanent impact;
3. queue-aware slippage models;
4. intraday volume curve execution;
5. capacity under volatility targeting;
6. alpha decay by regime;
7. turnover-aware optimisation;
8. execution scheduling for futures;
9. live TCA calibration;
10. Chinese futures capacity with night sessions, price limits, tick size, margin and exchange-specific liquidity.

## 31. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_11_live_shadow_mode_monitoring**  
   Compare expected alpha decay and cost with paper/live fills.

2. **05_12_full_research_to_production_checklist**  
   Final phase checklist combining costs, capacity and validation.

3. **06_06_execution_algorithms_twap_vwap_pov**  
   Turn capacity limits into execution algorithms.

4. **06_07_market_impact_model_calibration**  
   Estimate impact from historical fills.

5. **07_06_research_governance_capstone**  
   Combine turnover, PBO, WRC, DSR and live monitoring.

## 32. Summary

This notebook implemented turnover, capacity and alpha-decay analysis.

It showed how to:

1. simulate market and liquidity data;
2. build slow and fast alpha signals;
3. convert signals into weights;
4. calculate one-way and two-way turnover;
5. estimate holding-period proxies;
6. measure signal autocorrelation;
7. estimate information coefficient decay;
8. estimate alpha half-life;
9. model spread, slippage and impact costs;
10. compute cost-adjusted performance;
11. attribute trading costs by asset;
12. build capacity curves;
13. estimate breakeven capacity;
14. identify asset-level capacity bottlenecks;
15. model execution delay and alpha decay;
16. decompose turnover by signal component;
17. compare rebalance frequency trade-offs;
18. stress capacity during liquidity shocks;
19. build a capacity dashboard;
20. create governance flags and audit checks;
21. save outputs and manifests.

The key computational takeaway:

> Capacity analysis recomputes participation, impact, costs and net returns as AUM changes.

The key financial takeaway:

> The best alpha is not necessarily the highest IC signal; it is the signal whose alpha survives turnover, delay and market impact at the capital you actually want to run.

## 33. Further reading

- Almgren and Chriss on optimal execution.
- Grinold and Kahn, *Active Portfolio Management.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- López de Prado, *Advances in Financial Machine Learning.*
- Institutional transaction-cost-analysis literature on capacity, turnover and implementation shortfall.